# 13c CLIP-SENet CityFlowV2 Features

Extract CLIP-SENet v6 BNNeck features for every non-interpolated CityFlowV2 tracked detection in the 10a validation tracking output. Outputs are per-camera NPZ files plus a global manifest for downstream score fusion with TransReID 09v in Stage 4.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Match the CLIP-SENet training kernel environment and keep P100 (sm_60) compatible.
!pip install -q --upgrade torch==2.4.1+cu124 torchvision==0.19.1+cu124 --index-url https://download.pytorch.org/whl/cu124
!pip install -q --upgrade open_clip_torch==2.30.0 timm==1.0.11
!pip install -q pretrainedmodels==0.7.4 tqdm

import sys, os, time, json, random, hashlib, shutil, tarfile, re
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
print("python:", sys.version)
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "cuda_arch:", torch.cuda.get_arch_list() if torch.cuda.is_available() else None)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

In [ ]:
"""CLIP-SENet architecture for vehicle re-identification.

Milestone M1 covers only the model port and local forward-pass smoke tests.
Training losses, camera/viewpoint embeddings, and Kaggle integration are
intentionally deferred.
"""

from __future__ import annotations

from dataclasses import dataclass

import logging
import torch
import torch.nn as nn
import torch.nn.functional as F

logger = logging.getLogger(__name__)
if not logger.handlers:
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(_h)
    logger.setLevel(logging.INFO)


@dataclass(frozen=True)
class LoadedBackboneInfo:
    """Describes the backbone variant that was successfully loaded."""

    family: str
    model_name: str
    pretrained_tag: str | None = None


class AFEMBlock(nn.Module):
    """Adaptive Fine-grained Enhancement Module.

    This implements the paper's ambiguous Eq. (4) using the `(G + 1)`
    interpretation: `G` grouped weighted residual chunks plus one identity
    residual path. Set `residual_mode="sum_only"` to drop the identity term and
    return only the weighted grouped sum.
    """

    def __init__(
        self,
        in_dim: int = 2048,
        out_dim: int = 2048,
        num_groups: int = 32,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        if out_dim % num_groups != 0:
            raise ValueError(
                f"AFEM out_dim={out_dim} must be divisible by num_groups={num_groups}"
            )
        if residual_mode not in {"grouped_identity", "sum_only"}:
            raise ValueError(
                "residual_mode must be 'grouped_identity' or 'sum_only'"
            )

        self.in_dim = in_dim
        self.out_dim = out_dim
        self.num_groups = num_groups
        self.group_dim = out_dim // num_groups
        self.residual_mode = residual_mode

        self.shared = nn.Sequential(
            nn.Linear(in_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim),
            nn.ReLU(inplace=True),
        )
        self.group_weights = nn.Parameter(torch.randn(num_groups, self.group_dim))

        self._reset_parameters()

    def _reset_parameters(self) -> None:
        linear = self.shared[0]
        nn.init.kaiming_normal_(linear.weight, mode="fan_out")
        bn = self.shared[1]
        nn.init.ones_(bn.weight)
        nn.init.zeros_(bn.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.shared(x)
        grouped = h.view(h.shape[0], self.num_groups, self.group_dim)
        weighted = grouped * self.group_weights.unsqueeze(0)
        enhanced = weighted.reshape(h.shape[0], self.out_dim)
        if self.residual_mode == "sum_only":
            return enhanced
        return h + enhanced


class _ResNetFeatureWrapper(nn.Module):
    """Wrap torchvision-style ResNet backbones to expose pooled 2048-d features."""

    def __init__(self, model: nn.Module):
        super().__init__()
        self.conv1 = model.conv1
        self.bn1 = model.bn1
        self.relu = model.relu
        self.maxpool = model.maxpool
        self.layer1 = model.layer1
        self.layer2 = model.layer2
        self.layer3 = model.layer3
        self.layer4 = model.layer4
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return torch.flatten(x, 1)


class ResNet101IBNBranch(nn.Module):
    """Appearance branch backed by real ResNet101 IBN-a with deterministic fallbacks."""

    _IBN_MODEL = "resnet101_ibn_a"
    _FALLBACK_MODEL = "resnet101"

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.output_dim = 2048
        self.backbone: nn.Module
        self.loaded_backbone: LoadedBackboneInfo

        for loader in (
            self._load_pretrainedmodels_ibn,
            self._load_torch_hub_ibn,
            self._load_timm_ibn,
            self._load_timm_plain,
        ):
            loaded = loader(pretrained=pretrained)
            if loaded is None:
                continue
            self.backbone, self.loaded_backbone = loaded
            logger.info(
                "Appearance branch loaded via '%s' model='%s' pretrained_tag='%s'",
                self.loaded_backbone.family,
                self.loaded_backbone.model_name,
                self.loaded_backbone.pretrained_tag,
            )
            return

        raise ImportError(
            "Unable to load appearance backbone via pretrainedmodels, torch.hub, or timm"
        )

    def _load_pretrainedmodels_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import pretrainedmodels
        except ImportError:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' is unavailable; trying torch.hub"
            )
            return None

        constructor = getattr(pretrainedmodels, self._IBN_MODEL, None)
        if constructor is None:
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' has no '%s' entry; trying torch.hub",
                self._IBN_MODEL,
            )
            return None

        pretrained_tag = "imagenet" if pretrained else None
        try:
            raw_model = constructor(pretrained=pretrained_tag)
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'pretrainedmodels' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "last_linear"):
            raw_model.last_linear = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="pretrainedmodels",
            model_name=self._IBN_MODEL,
            pretrained_tag=pretrained_tag or "random_init",
        )

    def _load_torch_hub_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            raw_model = torch.hub.load(
                "XingangPan/IBN-Net",
                self._IBN_MODEL,
                pretrained=pretrained,
                trust_repo=True,
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'torch.hub' failed for '{}': {}",
                self._IBN_MODEL,
                exc,
            )
            return None

        if hasattr(raw_model, "fc"):
            raw_model.fc = nn.Identity()
        backbone = _ResNetFeatureWrapper(raw_model)
        return backbone, LoadedBackboneInfo(
            family="torch.hub",
            model_name=self._IBN_MODEL,
            pretrained_tag="official_pretrained" if pretrained else "random_init",
        )

    def _load_timm_ibn(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        available = set(timm.list_models())
        if self._IBN_MODEL not in available:
            logger.warning(
                "Appearance branch loader 'timm' has no '%s' entry; trying plain '%s'",
                self._IBN_MODEL,
                self._FALLBACK_MODEL,
            )
            return None

        try:
            backbone = timm.create_model(
                self._IBN_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for '%s': %s",
                self._IBN_MODEL,
                exc,
            )
            return None

        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._IBN_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def _load_timm_plain(
        self, pretrained: bool
    ) -> tuple[nn.Module, LoadedBackboneInfo] | None:
        try:
            import timm
        except ImportError as exc:
            raise ImportError("timm is required for ResNet101IBNBranch fallbacks") from exc

        try:
            backbone = timm.create_model(
                self._FALLBACK_MODEL,
                pretrained=pretrained,
                num_classes=0,
                global_pool="avg",
            )
        except Exception as exc:  # noqa: BLE001 - keep fallback chain moving
            logger.warning(
                "Appearance branch loader 'timm' failed for plain '%s': %s",
                self._FALLBACK_MODEL,
                exc,
            )
            return None

        logger.warning(
            "Appearance branch fell back to plain '%s' because no IBN-a loader succeeded",
            self._FALLBACK_MODEL,
        )
        return backbone, LoadedBackboneInfo(
            family="timm",
            model_name=self._FALLBACK_MODEL,
            pretrained_tag="timm_pretrained" if pretrained else "random_init",
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.backbone(x)
        if out.ndim != 2:
            raise RuntimeError(
                f"Appearance branch expected pooled 2D output, got shape {tuple(out.shape)}"
            )
        return out


class TinyCLIPImageBranch(nn.Module):
    """Semantic branch that loads TinyCLIP with a deterministic fallback chain."""

    _OPEN_CLIP_CHAIN = (
        {
            "model_name": "hf-hub:wkcn/TinyCLIP-ViT-45M-32-Text-21M-LAION400M",
            "pretrained_tag": None,
        },
        {
            "model_name": "TinyCLIP-ViT-40M-32-Text-19M",
            "pretrained_tag": "laion400m_e32",
        },
    )
    _TIMM_TINYCLIP_CHAIN = (
        "vit_medium_patch32_clip_224.tinyclip_laion400m",
    )
    _LAST_RESORT_OPEN_CLIP = ("ViT-B-32", "openai")

    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.provider = ""
        self.model = None
        self.loaded_backbone: LoadedBackboneInfo | None = None
        last_error = self._try_load_open_clip(pretrained=pretrained)
        if self.model is None:
            last_error = self._try_load_timm_tinyclip(pretrained=pretrained) or last_error
        if self.model is None:
            last_error = self._try_load_open_clip_last_resort(pretrained=pretrained) or last_error

        if self.model is None or self.loaded_backbone is None:
            raise RuntimeError(
                "Unable to load any TinyCLIP/OpenCLIP visual backbone"
            ) from last_error

        self.image_size = self._infer_image_size(self.model)

    def _try_load_open_clip(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for candidate in self._OPEN_CLIP_CHAIN:
            model_name = candidate["model_name"]
            pretrained_tag = candidate["pretrained_tag"]
            try:
                if pretrained_tag is None:
                    model, _, _ = open_clip.create_model_and_transforms(model_name)
                else:
                    model, _, _ = open_clip.create_model_and_transforms(
                        model_name,
                        pretrained=pretrained_tag if pretrained else None,
                    )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP load failed for model='%s' pretrained='%s': %s",
                    model_name,
                    pretrained_tag or "hf-hub-default",
                    exc,
                )
                continue

            self.model = model
            self.provider = "open_clip"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag=pretrained_tag if pretrained else "random_init",
            )
            self.output_dim = self._infer_open_clip_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
                model_name,
                pretrained_tag if pretrained_tag is not None and pretrained else "hf-hub-default",
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_timm_tinyclip(self, pretrained: bool) -> Exception | None:
        try:
            import timm
        except ImportError as exc:
            return exc

        last_error: Exception | None = None
        for model_name in self._TIMM_TINYCLIP_CHAIN:
            try:
                model = timm.create_model(
                    model_name,
                    pretrained=pretrained,
                    num_classes=0,
                )
            except Exception as exc:  # noqa: BLE001 - preserve fallback chain context
                last_error = exc
                logger.warning(
                    "TinyCLIP-equivalent timm load failed for model='%s': %s",
                    model_name,
                    exc,
                )
                continue

            self.model = model
            self.provider = "timm"
            self.loaded_backbone = LoadedBackboneInfo(
                family="semantic",
                model_name=model_name,
                pretrained_tag="timm_pretrained" if pretrained else "random_init",
            )
            self.output_dim = self._infer_timm_output_dim(model)
            logger.info(
                "TinyCLIP branch loaded model='%s' via timm output_dim=%s",
                model_name,
                self.output_dim,
            )
            return None

        return last_error

    def _try_load_open_clip_last_resort(self, pretrained: bool) -> Exception | None:
        try:
            import open_clip
        except ImportError as exc:
            return exc

        model_name, pretrained_tag = self._LAST_RESORT_OPEN_CLIP
        try:
            model, _, _ = open_clip.create_model_and_transforms(
                model_name,
                pretrained=pretrained_tag if pretrained else None,
            )
        except Exception as exc:  # noqa: BLE001 - explicit last resort context
            logger.warning(
                "OpenCLIP last resort load failed for model='%s' pretrained='%s': %s",
                model_name,
                pretrained_tag,
                exc,
            )
            return exc

        self.model = model
        self.provider = "open_clip"
        self.loaded_backbone = LoadedBackboneInfo(
            family="semantic",
            model_name=model_name,
            pretrained_tag=pretrained_tag if pretrained else "random_init",
        )
        self.output_dim = self._infer_open_clip_output_dim(model)
        logger.info(
            "TinyCLIP branch loaded model='%s' pretrained='%s' via open_clip output_dim=%s",
            model_name,
            pretrained_tag if pretrained else "random_init",
            self.output_dim,
        )
        return None

    @staticmethod
    def _infer_open_clip_output_dim(model: nn.Module) -> int:
        visual = getattr(model, "visual", None)
        output_dim = getattr(visual, "output_dim", None)
        if isinstance(output_dim, int):
            return output_dim

        visual_proj = getattr(model, "visual_projection", None)
        if isinstance(visual_proj, torch.Tensor) and visual_proj.ndim == 2:
            return int(visual_proj.shape[-1])

        visual_proj = getattr(visual, "proj", None)
        if isinstance(visual_proj, torch.Tensor):
            if visual_proj.ndim == 1:
                return int(visual_proj.shape[0])
            if visual_proj.ndim == 2:
                return int(visual_proj.shape[-1])

        raise RuntimeError("Could not infer TinyCLIP visual output dimension")

    @staticmethod
    def _infer_timm_output_dim(model: nn.Module) -> int:
        output_dim = getattr(model, "num_features", None)
        if isinstance(output_dim, int):
            return output_dim
        raise RuntimeError("Could not infer timm TinyCLIP visual output dimension")

    @staticmethod
    def _infer_image_size(model: nn.Module) -> tuple[int, int]:
        pretrained_cfg = getattr(model, "pretrained_cfg", None)
        if isinstance(pretrained_cfg, dict):
            input_size = pretrained_cfg.get("input_size")
            if isinstance(input_size, tuple) and len(input_size) == 3:
                return (int(input_size[-2]), int(input_size[-1]))

        visual = getattr(model, "visual", None)
        image_size = getattr(visual, "image_size", None)
        if isinstance(image_size, int):
            return (image_size, image_size)
        if isinstance(image_size, tuple) and len(image_size) == 2:
            return (int(image_size[0]), int(image_size[1]))
        return (224, 224)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if tuple(x.shape[-2:]) != self.image_size:
            x = F.interpolate(
                x,
                size=self.image_size,
                mode="bilinear",
                align_corners=False,
            )
        if self.provider == "open_clip":
            features = self.model.encode_image(x, normalize=False)
        else:
            features = self.model(x)
        if features.ndim != 2:
            raise RuntimeError(
                f"TinyCLIP branch expected 2D image features, got shape {tuple(features.shape)}"
            )
        return features


class CLIPSENet(nn.Module):
    """CLIP-SENet with a CNN appearance branch and a CLIP semantic branch."""

    def __init__(
        self,
        num_classes: int,
        embed_dim: int = 2048,
        afem_groups: int = 32,
        feat_dim_appearance: int = 2048,
        feat_dim_semantic: int = 512,
        dropout: float = 0.0,
        appearance_pretrained: bool = True,
        semantic_pretrained: bool = True,
        residual_mode: str = "grouped_identity",
    ):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim

        self.appearance_branch = ResNet101IBNBranch(pretrained=appearance_pretrained)
        self.semantic_branch = TinyCLIPImageBranch(pretrained=semantic_pretrained)

        detected_app_dim = self.appearance_branch.output_dim
        detected_sem_dim = self.semantic_branch.output_dim
        if feat_dim_appearance != detected_app_dim:
            logger.warning(
                "Requested feat_dim_appearance=%s but backbone reports %s. Using detected dim.",
                feat_dim_appearance,
                detected_app_dim,
            )
        if feat_dim_semantic != detected_sem_dim:
            logger.warning(
                "Requested feat_dim_semantic=%s but backbone reports %s. Using detected dim.",
                feat_dim_semantic,
                detected_sem_dim,
            )

        self.feat_dim_appearance = detected_app_dim
        self.feat_dim_semantic = detected_sem_dim
        self.fusion_fc = nn.Linear(
            self.feat_dim_appearance + self.feat_dim_semantic,
            embed_dim,
            bias=False,
        )
        self.afem = AFEMBlock(
            in_dim=embed_dim,
            out_dim=embed_dim,
            num_groups=afem_groups,
            residual_mode=residual_mode,
        )
        self.bnneck = nn.BatchNorm1d(embed_dim)
        self.bnneck.bias.requires_grad_(False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.classifier = nn.Linear(embed_dim, num_classes, bias=False)

        nn.init.kaiming_normal_(self.fusion_fc.weight, mode="fan_out")
        nn.init.normal_(self.classifier.weight, std=0.001)

        self.loaded_resnext_model = self.appearance_branch.loaded_backbone.model_name
        self.loaded_tinyclip_model = self.semantic_branch.loaded_backbone.model_name

    def forward(self, x: torch.Tensor):
        f_app = self.appearance_branch(x)
        f_sem = self.semantic_branch(x)
        t_u = self.fusion_fc(torch.cat([f_app, f_sem], dim=1))
        t_s_prime = self.afem(t_u)
        t = t_u + t_s_prime
        t_bn = self.bnneck(t)
        t_bn_normalized = F.normalize(t_bn, p=2, dim=1)

        if self.training:
            logits = self.classifier(self.dropout(t_bn))
            return t_bn_normalized, logits

        return t_bn_normalized


def build_clip_senet(num_classes: int, **kwargs) -> CLIPSENet:
    """Build a CLIP-SENet model for M1 architecture validation."""

    return CLIPSENet(num_classes=num_classes, **kwargs)

In [ ]:
import json
from pathlib import Path

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_NUM_CLASSES = 575
CHECKPOINT_CANDIDATE_NAMES = ("best.pth", "vehicle_clip_senet_veri776.pth")
PREFERRED_EXACT = Path("/kaggle/input/notebooks/yahiaakhalafallah/13-clip-senet-train/checkpoints/best.pth")
PREFERRED_SOURCE_SLUG = "13-clip-senet-train"


def torch_load(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def discover_clip_senet_checkpoint() -> Path:
    if PREFERRED_EXACT.is_file():
        print(f"Using preferred checkpoint: {PREFERRED_EXACT}")
        return PREFERRED_EXACT

    candidates = []
    for root in sorted(Path("/kaggle/input").rglob("*")):
        if not root.is_dir():
            continue
        for name in CHECKPOINT_CANDIDATE_NAMES:
            candidate = root / name
            if candidate.is_file():
                score = 0
                candidate_str = str(candidate)
                if PREFERRED_SOURCE_SLUG in candidate_str:
                    score += 100
                if "checkpoints" in candidate.parts:
                    score += 10
                if name == "best.pth":
                    score += 5
                candidates.append((score, candidate))
    if not candidates:
        raise FileNotFoundError(
            "Could not find CLIP-SENet checkpoint under /kaggle/input. "
            "Expected chained kernel source output from yahiaakhalafallah/13-clip-senet-train."
        )
    candidates.sort(key=lambda item: (-item[0], str(item[1])))
    print("Checkpoint candidates:")
    for score, path in candidates[:10]:
        print(f"  score={score:3d}  {path}")
    return candidates[0][1]


CHECKPOINT_PATH = discover_clip_senet_checkpoint()
payload = torch_load(CHECKPOINT_PATH)
if isinstance(payload, dict) and "model_state" in payload:
    state_dict = payload["model_state"]
    checkpoint_kind = "payload:model_state"
elif isinstance(payload, dict) and "model" in payload and isinstance(payload["model"], dict):
    state_dict = payload["model"]
    checkpoint_kind = "payload:model"
elif isinstance(payload, dict) and payload and all(hasattr(value, "shape") for value in payload.values()):
    state_dict = payload
    checkpoint_kind = "state_dict"
else:
    raise TypeError(f"Unsupported checkpoint format at {CHECKPOINT_PATH}: {type(payload).__name__}")

classifier_weight = state_dict.get("classifier.weight")
inferred_num_classes = int(classifier_weight.shape[0]) if classifier_weight is not None else DEFAULT_NUM_CLASSES
model = build_clip_senet(num_classes=inferred_num_classes).to(DEVICE)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
if missing_keys or unexpected_keys:
    print("Missing keys:", missing_keys)
    print("Unexpected keys:", unexpected_keys)
    raise RuntimeError("Checkpoint load was not strict; aborting feature extraction setup")
model.eval()
FEATURE_DIM = int(getattr(model, "embed_dim", 2048))

RUN_INFO = {
    "device": DEVICE,
    "checkpoint_path": str(CHECKPOINT_PATH),
    "checkpoint_kind": checkpoint_kind,
    "num_classes": inferred_num_classes,
    "feature_dim": FEATURE_DIM,
    "loaded_tinyclip_model": getattr(model, "loaded_tinyclip_model", None),
    "loaded_appearance_model": getattr(model, "loaded_resnext_model", None),
}
print(json.dumps(RUN_INFO, indent=2))

In [ ]:
import json
import shutil
import tarfile
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
EXTRACT_ROOT = Path("/tmp/mtmc_10a_stage_outputs")
OUTPUT_DIR = WORK_ROOT / "clip_senet_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CAM_RE = re.compile(r"^S\d{2}_c\d{3}$")
VIDEO_EXTS = {".mp4", ".avi", ".mkv", ".mov"}


def looks_like_cityflow_root(path: Path) -> bool:
    if not path.is_dir():
        return False
    split_dirs = [path / split for split in ("validation", "train", "test")]
    return any(split_dir.is_dir() for split_dir in split_dirs)


def discover_cityflow_root() -> Path:
    explicit = [
        Path("/kaggle/input/data-aicity-2023-track-2"),
        Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
    ]
    for candidate in explicit:
        if looks_like_cityflow_root(candidate):
            return candidate
        if candidate.exists():
            for nested in candidate.rglob("*"):
                if looks_like_cityflow_root(nested):
                    return nested
    scored = []
    for candidate in INPUT_ROOT.rglob("*"):
        if looks_like_cityflow_root(candidate):
            score = 0
            text = str(candidate).lower()
            if "data-aicity-2023-track-2" in text:
                score += 100
            if "cityflow" in text or "aic" in text:
                score += 20
            scored.append((score, candidate))
    if not scored:
        raise FileNotFoundError("CityFlowV2 root not found under /kaggle/input")
    scored.sort(key=lambda item: (-item[0], str(item[1])))
    return scored[0][1]


def camera_id_from_scene_cam(scene_dir: Path, cam_dir: Path) -> str:
    return f"{scene_dir.name}_{cam_dir.name}"


def discover_cityflow_videos(cityflow_root: Path):
    videos = {}
    split_by_camera = {}
    for split_name in ("validation", "train", "test"):
        split_dir = cityflow_root / split_name
        if not split_dir.is_dir():
            continue
        for scene_dir in sorted(split_dir.iterdir()):
            if not scene_dir.is_dir():
                continue
            for cam_dir in sorted(scene_dir.iterdir()):
                if not cam_dir.is_dir():
                    continue
                cam_id = camera_id_from_scene_cam(scene_dir, cam_dir)
                cam_videos = [p for p in cam_dir.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTS]
                if not cam_videos:
                    cam_videos = [p for p in cam_dir.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS]
                if cam_videos:
                    videos[cam_id] = sorted(cam_videos)[0]
                    split_by_camera[cam_id] = split_name
    if not videos:
        raise FileNotFoundError(f"No camera videos found below {cityflow_root}")
    return videos, split_by_camera


def extract_10a_checkpoint_if_needed() -> Path:
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    existing = list(EXTRACT_ROOT.rglob("tracklets_*.json"))
    if existing:
        return EXTRACT_ROOT

    preferred_roots = [
        Path("/kaggle/input/notebooks/yahiaakhalafallah/mtmc-10a-stages-0-2"),
        Path("/kaggle/input/mtmc-10a-stages-0-2"),
    ]
    checkpoint_candidates = []
    for root in preferred_roots:
        if root.exists():
            checkpoint_candidates.extend(root.rglob("checkpoint.tar.gz"))
            checkpoint_candidates.extend(root.rglob("*.tar.gz"))
    if not checkpoint_candidates:
        for candidate in INPUT_ROOT.rglob("*.tar.gz"):
            text = str(candidate).lower()
            if "10a" in text or "stages-0-2" in text or "mtmc-10a" in text:
                checkpoint_candidates.append(candidate)

    checkpoint_candidates = sorted(set(checkpoint_candidates), key=lambda p: ("checkpoint.tar.gz" not in p.name, str(p)))
    if checkpoint_candidates:
        checkpoint_path = checkpoint_candidates[0]
        print(f"Extracting 10a checkpoint: {checkpoint_path}")
        with tarfile.open(checkpoint_path, "r:gz") as tar:
            tar.extractall(EXTRACT_ROOT)
        return EXTRACT_ROOT

    direct_tracklets = []
    for root in preferred_roots:
        if root.exists():
            direct_tracklets.extend(root.rglob("tracklets_*.json"))
    if direct_tracklets:
        return direct_tracklets[0].parent

    raise FileNotFoundError(
        "Could not find 10a tracking output. Expected kernel source "
        "yahiaakhalafallah/mtmc-10a-stages-0-2 with checkpoint.tar.gz or tracklets_*.json."
    )


def discover_stage1_dir(root: Path) -> Path:
    candidates = []
    for path in root.rglob("tracklets_*.json"):
        if path.is_file():
            count = len(list(path.parent.glob("tracklets_*.json")))
            score = count
            if path.parent.name == "stage1":
                score += 100
            candidates.append((score, path.parent))
    if not candidates:
        raise FileNotFoundError(f"No tracklets_*.json files found below {root}")
    candidates.sort(key=lambda item: (-item[0], str(item[1])))
    return candidates[0][1]


CITYFLOW_ROOT = discover_cityflow_root()
CAMERA_VIDEOS, CAMERA_SPLITS = discover_cityflow_videos(CITYFLOW_ROOT)
TRACKING_ROOT = extract_10a_checkpoint_if_needed()
STAGE1_DIR = discover_stage1_dir(TRACKING_ROOT)
TRACKLET_FILES = sorted(STAGE1_DIR.glob("tracklets_*.json"))

print("CITYFLOW_ROOT:", CITYFLOW_ROOT)
print("TRACKING_ROOT:", TRACKING_ROOT)
print("STAGE1_DIR:", STAGE1_DIR)
print("TRACKLET_FILES:", len(TRACKLET_FILES))
for path in TRACKLET_FILES:
    cam_id = path.stem.replace("tracklets_", "")
    print(f"  {cam_id}: split={CAMERA_SPLITS.get(cam_id, 'unknown')} video={CAMERA_VIDEOS.get(cam_id)}")

In [ ]:
from PIL import Image
import torch
import torchvision.transforms as T
from tqdm.auto import tqdm

IMAGE_SIZE = (320, 320)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
BATCH_SIZE = 64

transform = T.Compose([
    T.Resize(IMAGE_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def apply_cityflow_stage0_preprocess(frame_bgr: np.ndarray) -> np.ndarray:
    # Mirror the CityFlowV2 stage0 CLAHE setting used by the 10a feature kernel.
    lab = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l_channel = clahe.apply(l_channel)
    return cv2.cvtColor(cv2.merge((l_channel, a_channel, b_channel)), cv2.COLOR_LAB2BGR)


def crop_xyxy(frame_bgr: np.ndarray, bbox):
    height, width = frame_bgr.shape[:2]
    x1, y1, x2, y2 = [float(value) for value in bbox]
    x1 = int(max(0, min(width - 1, round(x1))))
    y1 = int(max(0, min(height - 1, round(y1))))
    x2 = int(max(0, min(width, round(x2))))
    y2 = int(max(0, min(height, round(y2))))
    if x2 <= x1 or y2 <= y1:
        return None
    crop_bgr = frame_bgr[y1:y2, x1:x2]
    if crop_bgr.size == 0:
        return None
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(crop_rgb)


def read_frame(cap: cv2.VideoCapture, frame_id: int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
    ok, frame = cap.read()
    if not ok or frame is None:
        return None
    return frame


def feature_batch(images):
    if not images:
        return np.empty((0, FEATURE_DIM), dtype=np.float32)
    tensor = torch.stack([transform(image) for image in images], dim=0).to(DEVICE, non_blocking=True)
    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            features = model(tensor)
        features = torch.nn.functional.normalize(features.float(), p=2, dim=1)
    return features.cpu().numpy().astype(np.float32)

print("Transform:", IMAGE_SIZE, IMAGENET_MEAN, IMAGENET_STD)
print("Batch size:", BATCH_SIZE)

In [ ]:
global_index = []
manifest = {
    "run_info": RUN_INFO,
    "cityflow_root": str(CITYFLOW_ROOT),
    "stage1_dir": str(STAGE1_DIR),
    "output_dir": str(OUTPUT_DIR),
    "image_size": list(IMAGE_SIZE),
    "normalization": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
    "per_camera": {},
    "index": global_index,
}

total_detections = 0
skipped_zero_conf = 0
skipped_missing_video = 0
skipped_missing_frame = 0
skipped_bad_crop = 0
feature_dim_seen = None

for tracklet_file in TRACKLET_FILES:
    camera_id = tracklet_file.stem.replace("tracklets_", "")
    video_path = CAMERA_VIDEOS.get(camera_id)
    if video_path is None:
        print(f"SKIP {camera_id}: no CityFlowV2 video found")
        skipped_missing_video += 1
        continue

    tracklets = json.loads(tracklet_file.read_text())
    records_by_frame = defaultdict(list)
    det_counter = 0
    for tracklet in tracklets:
        track_id = int(tracklet["track_id"])
        for frame in tracklet.get("frames", []):
            confidence = float(frame.get("confidence", 1.0))
            if confidence <= 0.0:
                skipped_zero_conf += 1
                continue
            record = {
                "det_id": det_counter,
                "track_id": track_id,
                "frame_id": int(frame["frame_id"]),
                "bbox": [float(value) for value in frame["bbox"]],
                "confidence": confidence,
            }
            records_by_frame[record["frame_id"]].append(record)
            det_counter += 1

    embeddings = []
    frame_ids = []
    bbox_xyxy = []
    track_ids = []
    det_ids = []
    confidences = []
    pending_images = []
    pending_records = []

    def flush_batch():
        nonlocal_feature_count = len(pending_images)
        if nonlocal_feature_count == 0:
            return
        batch_embeddings = feature_batch(pending_images)
        embeddings.append(batch_embeddings)
        for local_record, embedding in zip(pending_records, batch_embeddings):
            local_index = len(frame_ids)
            global_feature_index = total_detections + local_index
            frame_ids.append(local_record["frame_id"])
            bbox_xyxy.append(local_record["bbox"])
            track_ids.append(local_record["track_id"])
            det_ids.append(local_record["det_id"])
            confidences.append(local_record["confidence"])
            global_index.append({
                "camera_id": camera_id,
                "frame_id": int(local_record["frame_id"]),
                "det_id": int(local_record["det_id"]),
                "track_id": int(local_record["track_id"]),
                "feature_index": int(global_feature_index),
                "local_index": int(local_index),
                "bbox_xyxy": [float(value) for value in local_record["bbox"]],
            })
        pending_images.clear()
        pending_records.clear()

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Cannot open video for {camera_id}: {video_path}")

    try:
        progress_total = sum(len(items) for items in records_by_frame.values())
        pbar = tqdm(total=progress_total, desc=f"{camera_id}")
        for frame_id in sorted(records_by_frame):
            raw_frame = read_frame(cap, frame_id)
            if raw_frame is None:
                skipped_missing_frame += len(records_by_frame[frame_id])
                pbar.update(len(records_by_frame[frame_id]))
                continue
            frame_bgr = apply_cityflow_stage0_preprocess(raw_frame)
            for record in records_by_frame[frame_id]:
                crop = crop_xyxy(frame_bgr, record["bbox"])
                if crop is None:
                    skipped_bad_crop += 1
                    pbar.update(1)
                    continue
                pending_images.append(crop)
                pending_records.append(record)
                if len(pending_images) >= BATCH_SIZE:
                    flush_batch()
                pbar.update(1)
        flush_batch()
        pbar.close()
    finally:
        cap.release()

    if embeddings:
        embeddings_arr = np.concatenate(embeddings, axis=0).astype(np.float32)
    else:
        embeddings_arr = np.empty((0, FEATURE_DIM), dtype=np.float32)
    frame_ids_arr = np.asarray(frame_ids, dtype=np.int64)
    bbox_arr = np.asarray(bbox_xyxy, dtype=np.float32).reshape(-1, 4)
    track_ids_arr = np.asarray(track_ids, dtype=np.int64)
    det_ids_arr = np.asarray(det_ids, dtype=np.int64)
    confidences_arr = np.asarray(confidences, dtype=np.float32)

    if embeddings_arr.size:
        norms = np.linalg.norm(embeddings_arr, axis=1)
        print(f"{camera_id}: norm min/mean/max = {norms.min():.6f}/{norms.mean():.6f}/{norms.max():.6f}")
        feature_dim_seen = int(embeddings_arr.shape[1])

    output_path = OUTPUT_DIR / f"{camera_id}.npz"
    np.savez_compressed(
        output_path,
        embeddings=embeddings_arr,
        frame_ids=frame_ids_arr,
        bbox_xyxy=bbox_arr,
        track_ids=track_ids_arr,
        det_ids=det_ids_arr,
        confidences=confidences_arr,
    )
    file_size_mb = output_path.stat().st_size / 1024**2
    manifest["per_camera"][camera_id] = {
        "file": str(output_path),
        "split": CAMERA_SPLITS.get(camera_id),
        "video": str(video_path),
        "tracklet_file": str(tracklet_file),
        "count": int(embeddings_arr.shape[0]),
        "feature_dim": int(embeddings_arr.shape[1]) if embeddings_arr.ndim == 2 else FEATURE_DIM,
        "offset": int(total_detections),
        "file_size_mb": round(file_size_mb, 3),
    }
    total_detections += int(embeddings_arr.shape[0])
    print(f"Saved {camera_id}: {embeddings_arr.shape} -> {output_path} ({file_size_mb:.1f} MB)")

manifest["summary"] = {
    "total_detections": int(total_detections),
    "total_cameras": int(len(manifest["per_camera"])),
    "feature_dim": int(feature_dim_seen or FEATURE_DIM),
    "skipped_zero_conf_interpolated": int(skipped_zero_conf),
    "skipped_missing_video_cameras": int(skipped_missing_video),
    "skipped_missing_frames": int(skipped_missing_frame),
    "skipped_bad_crops": int(skipped_bad_crop),
}
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Manifest:", manifest_path)
print(json.dumps(manifest["summary"], indent=2))

In [ ]:
total_size = 0.0
for path in sorted(OUTPUT_DIR.glob("*.npz")):
    size_mb = path.stat().st_size / 1024**2
    total_size += size_mb
    with np.load(path) as data:
        print(f"{path.name:16s} embeddings={data['embeddings'].shape} frames={data['frame_ids'].shape} size={size_mb:.1f} MB")
manifest_size_mb = (OUTPUT_DIR / "manifest.json").stat().st_size / 1024**2
print("-" * 72)
print(f"Total cameras    : {manifest['summary']['total_cameras']}")
print(f"Total detections : {manifest['summary']['total_detections']}")
print(f"Feature dim      : {manifest['summary']['feature_dim']}")
print(f"NPZ total size   : {total_size:.1f} MB")
print(f"Manifest size    : {manifest_size_mb:.1f} MB")
print(f"Output directory : {OUTPUT_DIR}")